--- Day 9: Movie Theater ---
You slide down the firepole in the corner of the playground and land in the North Pole base movie theater!

The movie theater has a big tile floor with an interesting pattern. Elves here are redecorating the theater by switching out some of the square tiles in the big grid they form. Some of the tiles are red; the Elves would like to find the largest rectangle that uses red tiles for two of its opposite corners. They even have a list of where the red tiles are located in the grid (your puzzle input).

For example:

7,1
11,1
11,7
9,7
9,5
2,5
2,3
7,3
Showing red tiles as # and other tiles as ., the above arrangement of red tiles would look like this:

..............
.......#...#..
..............
..#....#......
..............
..#......#....
..............
.........#.#..
..............
You can choose any two red tiles as the opposite corners of your rectangle; your goal is to find the largest rectangle possible.

For example, you could make a rectangle (shown as O) with an area of 24 between 2,5 and 9,7:

..............
.......#...#..
..............
..#....#......
..............
..OOOOOOOO....
..OOOOOOOO....
..OOOOOOOO.#..
..............
Or, you could make a rectangle with area 35 between 7,1 and 11,7:

..............
.......OOOOO..
.......OOOOO..
..#....OOOOO..
.......OOOOO..
..#....OOOOO..
.......OOOOO..
.......OOOOO..
..............
You could even make a thin rectangle with an area of only 6 between 7,3 and 2,3:

..............
.......#...#..
..............
..OOOOOO......
..............
..#......#....
..............
.........#.#..
..............
Ultimately, the largest rectangle you can make in this example has area 50. One way to do this is between 2,5 and 11,1:

..............
..OOOOOOOOOO..
..OOOOOOOOOO..
..OOOOOOOOOO..
..OOOOOOOOOO..
..OOOOOOOOOO..
..............
.........#.#..
..............
Using two red tiles as opposite corners, what is the largest area of any rectangle you can make?

In [1]:
with open('input.txt') as file:
    red_tiles = [tuple(int(coord) for coord in tile.split(',')) for tile in file]
len(red_tiles)

496

In [2]:
def area(tile_a, tile_b):
    a_x, a_y = tile_a
    b_x, b_y = tile_b
    d_x = abs(a_x - b_x) + 1
    d_y = abs(a_y - b_y) + 1
    return d_x * d_y

In [3]:
# brute-force:
max_area = 0
for ix, tile_a in enumerate(red_tiles):
    for tile_b in red_tiles[ix+1:]:
        if area(tile_a, tile_b) > max_area:
            max_area = area(tile_a, tile_b)
print(f'{max_area:e}')
max_area


4.765757e+09


4765757080

--- Part Two ---
The Elves just remembered: they can only switch out tiles that are red or green. So, your rectangle can only include red or green tiles.

In your list, every red tile is connected to the red tile before and after it by a straight line of green tiles. The list wraps, so the first red tile is also connected to the last red tile. Tiles that are adjacent in your list will always be on either the same row or the same column.

Using the same example as before, the tiles marked X would be green:

..............
.......#XXX#..
.......X...X..
..#XXXX#...X..
..X........X..
..#XXXXXX#.X..
.........X.X..
.........#X#..
..............
In addition, all of the tiles inside this loop of red and green tiles are also green. So, in this example, these are the green tiles:

..............
.......#XXX#..
.......XXXXX..
..#XXXX#XXXX..
..XXXXXXXXXX..
..#XXXXXX#XX..
.........XXX..
.........#X#..
..............
The remaining tiles are never red nor green.

The rectangle you choose still must have red tiles in opposite corners, but any other tiles it includes must now be red or green. This significantly limits your options.

For example, you could make a rectangle out of red and green tiles with an area of 15 between 7,3 and 11,1:

..............
.......OOOOO..
.......OOOOO..
..#XXXXOOOOO..
..XXXXXXXXXX..
..#XXXXXX#XX..
.........XXX..
.........#X#..
..............
Or, you could make a thin rectangle with an area of 3 between 9,7 and 9,5:

..............
.......#XXX#..
.......XXXXX..
..#XXXX#XXXX..
..XXXXXXXXXX..
..#XXXXXXOXX..
.........OXX..
.........OX#..
..............
The largest rectangle you can make in this example using only red and green tiles has area 24. One way to do this is between 9,5 and 2,3:

..............
.......#XXX#..
.......XXXXX..
..OOOOOOOOXX..
..OOOOOOOOXX..
..OOOOOOOOXX..
.........XXX..
.........#X#..
..............
Using two red tiles as opposite corners, what is the largest area of any rectangle you can make using only red and green tiles?

In [4]:
# let me make sure I understand the assignment. 
# First: Do all red tiles have exactly two "adjacent" tiles that share row or column?
dist_adjacent = {}
for a_x, a_y in red_tiles:
    num_adjacent = 0
    for b_x, b_y in red_tiles:
        if a_x == b_x or a_y == b_y:
            num_adjacent += 1
    num_adjacent -= 1 # correct for self as neighbor
    if num_adjacent == 4:
        print(f'{a_x, a_y=}')
    if num_adjacent in dist_adjacent:
        dist_adjacent[num_adjacent] += 1
    else:
        dist_adjacent[num_adjacent] = 1
dist_adjacent

a_x, a_y=(50434, 2033)
a_x, a_y=(51652, 2033)
a_x, a_y=(54080, 2033)
a_x, a_y=(55319, 2033)


{2: 492, 4: 4}

In [5]:
# Ok 492 red tiles have 2 adjacent tiles, and there is a weird cluster of 4 tiles that share a y-coordinate (2033). Let's move on towards mapping out the clusters. Is every tile connected or are there multiple islands of circular tilings?
# Let's try Union-find, which Claude recommended I use when reviewing my solution to yesterday's puzzle (though it might be overkill here since we don't merge over and over just once)

parent = {}
for tile in red_tiles:
    parent[tile] = tile

def find(node):
    while node != parent[node]:
        parent[node] = parent[parent[node]] # compress path
        node = parent[node]
    return node

def merge(a, b):
    pa, pb = find(a), find(b)
    if pa != pb:
        parent[pa] = pb


for ix, tile_a in enumerate(red_tiles):
    a_x, a_y = tile_a
    for tile_b in red_tiles[ix+1:]:
        b_x, b_y = tile_b
        if a_x == b_x or a_y == b_y:
            merge(tile_a, tile_b)

In [6]:
# Now lets unroll the clusters:
clusters = {}
for tile in red_tiles:
    cluster = find(tile)
    if cluster in clusters:
        clusters[cluster].append(tile)
    else:
        clusters[cluster] = [tile]
len(clusters)

1

In [7]:
# Guess it's just one big cluster! Also rereading, I see now how I misunderstood the assignment: Red tiles that are next to each other in the input list share a row or column :headslam:. Ok. That simplifies building the cluster a lot. I will go through the list one by one and build the map as we go. With each new line we can paint in all tiles between the two tiles. At the end we have a large grid with many lines. 
# 
# After that we have to paint in the loop. I'm not sure how to do that. I guess we can start with a random tile and give it color A, and fill in from there coloring all adjacents and their adjacents and so on with color A. Once we're done, we can check if any of the color A spots hit the edge of the grid (then it's outside) or not (inside). We can also check if starting with one of the empty tiles and filling that in we will have filled out the whole grid or not. 

# Actually maybe that is thinking way too complicated. What if we just look for the biggest squares as before, but with the added twist that no red tiles or green lines can be inside the area spanned by the rectangle. My intuition is that unless there are holes in the filled in space (donut etc) checking for red tiles and green lines should be sufficient to ensure that the whole space inside is filled in

In [8]:
green_tiles = set()

def make_green(x, y):
    if isinstance(x, tuple):
        x = range(x[0] + 1, x[1])
    else:
        x = [x]
    if isinstance(y, tuple):
        y = range(y[0] + 1, y[1])
    else:
        y = [y]

    for x_ in x:
        for y_ in y:
            green_tiles.add((x_, y_))

for (a_x, a_y), (b_x, b_y) in zip(red_tiles, red_tiles[1:]):
    if a_x == b_x:
            make_green(x=a_x, y=(min(a_y, b_y), max(a_y, b_y)))
    elif a_y == b_y:
            make_green(x=(min(a_x, b_x), max(a_x, b_x)), y=a_y)
    else:
        raise ValueError

In [19]:
len(green_tiles), next(iter(green_tiles))

(588507, (56131, 97958))

In [ ]:
# with some optimization help from Claude (check each forbidden tile directly rather than checking every point in the grid for membership in red_and_green)

red_and_green = green_tiles.union(red_tiles)
def edge_inside(a, b):
    a_x, a_y = a
    b_x, b_y = b
    lo_x, hi_x = min(a_x, b_x), max(a_x, b_x)
    lo_y, hi_y = min(a_y, b_y), max(a_y, b_y)
    for x, y in red_and_green:
        if lo_x < x < hi_x and lo_y < y < hi_y:
            return True
    return False
len(red_and_green), len(green_tiles)

(589003, 588507)

In [38]:
edge_inside((0, 97950),(56132, 99999))

True

In [40]:
max_area = 0
for ix, tile_a in enumerate(red_tiles):
    for tile_b in red_tiles[ix+1:]:
        if area(tile_a, tile_b) > max_area and not edge_inside(tile_a, tile_b):
            max_area = area(tile_a, tile_b)
            print(f'New largest tile found: Size {max_area}')
print(f'{max_area:e}')
max_area

New largest tile found: Size 1214
New largest tile found: Size 503810
New largest tile found: Size 33019758
New largest tile found: Size 84412989
New largest tile found: Size 151930938
New largest tile found: Size 152160183
New largest tile found: Size 201018285
New largest tile found: Size 236972775
New largest tile found: Size 1498673376
1.498673e+09


1498673376

That's the right answer! You are one gold star closer to decorating the North Pole.

You have completed Day 9! You can [Share] this victory or [Return to Your Advent Calendar].